# PD03 · Integrar EVA con DIVIPOLA

[Índice y distribución de las cuatro horas](../ejercicios/README.md) · [Guía de pandas](../docs/manual-pandas.md)

**Tiempo de trabajo: 70 minutos.** El objetivo es comprobar una unión territorial y calcular un rendimiento con el denominador adecuado.

| Símbolo | Uso |
|:---:|---|
| 📘 | Concepto o función de apoyo. |
| ✅ | Evidencia que debe quedar visible. |
| ⚠️ | Límite de interpretación. |

Esta plantilla se encuentra en `ejercicios/PD03_integracion_territorial.ipynb`. Antes de comenzar, se crea una copia en `soluciones/PD03_integracion_territorial.ipynb` dentro del repositorio de cada estudiante. El desarrollo y la entrega se realizan en esa copia; la plantilla de `ejercicios/` se conserva como enunciado. En JupyterLab se puede duplicar el archivo y mover la copia a `soluciones/`, manteniendo el nombre indicado; si ya existe una solución, se continúa sobre ella. Las instrucciones se presentan en celdas Markdown; las celdas de código marcadas como **Desarrollo** contienen el espacio de trabajo. Cada actividad incluye una celda para explicar el resultado.

Se utiliza el kernel del entorno `.venv` del curso. Los CSV ya deben estar descargados mediante `kit/00_datos.py`. La celda inicial prepara las rutas y carga únicamente las entradas de este ejercicio; el desarrollo del análisis corresponde al estudiante. Este notebook se ejecuta de forma independiente.

**Apoyo en el manual:** secciones 4, 7, 9, 10 y la receta de razón agregada de la sección 11.

## Ubicación de trabajo: la raíz del repositorio

📘 **La raíz del repositorio** es la carpeta principal de la copia del proyecto. En la instalación del curso se llama `big-data-postgraduate` y contiene `README.md`, `kit/`, `ejercicios/` y `soluciones/`:

```text
bigdata/
└── big-data-postgraduate/       ← raíz del repositorio
    ├── README.md
    ├── kit/
    ├── ejercicios/              ← enunciados y plantillas
    └── soluciones/              ← notebooks resueltos
```

**Desde la terminal de Ubuntu en WSL.** `pwd` muestra la carpeta actual, `cd` cambia de carpeta y `ls` muestra su contenido. Los siguientes comandos se ejecutan en la terminal, no en una celda Python:

```bash
pwd
cd /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate
pwd
ls
```

✅ Después de `cd`, el segundo `pwd` debe mostrar `/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate`. En la salida de `ls` deben aparecer `README.md`, `kit`, `ejercicios` y `soluciones`. La carpeta `bigdata` es la carpeta contenedora; la raíz del proyecto es su subcarpeta `big-data-postgraduate`.

La ruta anterior corresponde a la configuración de la guía WSL. Si el repositorio se clonó en otra ubicación, se sustituye por la ruta real de esa copia. Si `cd` informa que la carpeta no existe, se revisa esa ubicación antes de continuar.

**Desde una subcarpeta conocida.** `..` significa «carpeta superior». Se comprueba primero la ubicación con `pwd` y se utiliza la alternativa correspondiente:

| Ubicación actual, relativa al repositorio | Comando para volver a la raíz |
|---|---|
| `ejercicios/`, `soluciones/` o `kit/` | `cd ..` |
| `kit/Notebooks/` | `cd ../..` |

Al llegar a la raíz, se puede activar el entorno y abrir JupyterLab desde la terminal:

```bash
source .venv/bin/activate
jupyter lab
```

**Dentro de un notebook ya abierto.** La terminal y el kernel de Python mantienen sus propios directorios de trabajo. La celda de carga siguiente localiza la raíz y la guarda en `REPO`, aunque el notebook esté abierto desde `soluciones/` o `ejercicios/`. Después de ejecutar esa celda, el siguiente bloque puede ejecutarse en una celda Python para comprobar la ubicación y cambiar el directorio del kernel a la raíz:

```python
import os
from pathlib import Path

print("Directorio antes del cambio:", Path.cwd())
print("Raíz detectada:", REPO)
os.chdir(REPO)
print("Directorio después del cambio:", Path.cwd())
```

✅ `Path.cwd()` debe coincidir con `REPO` después del cambio. Esta operación cambia la base para interpretar rutas relativas; el archivo del notebook permanece en `soluciones/`. La carga de datos de estos ejercicios también funciona sin ese cambio porque construye las rutas a partir de `REPO`.

## Preparación

Se ejecuta la siguiente celda y se comprueba que el intérprete pertenece al entorno del curso y que pandas corresponde a la versión 2.2.3. Las rutas se calculan desde el repositorio, sin introducir rutas personales de Windows, WSL o macOS.

In [3]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

actual = Path.cwd().resolve()
REPO = next(
    (ruta for ruta in [actual, *actual.parents]
     if (ruta / "kit/fuentes.json").is_file()
     and (ruta / "kit/00_datos.py").is_file()),
    None,
)
if REPO is None:
    raise FileNotFoundError("El notebook debe abrirse dentro del repositorio del curso")

RAW = REPO / "kit/data/raw"
OUT = REPO / "kit/salidas/soluciones/PD03"
archivos = ['eva.csv', 'divipola.csv']
faltantes = [nombre for nombre in archivos if not (RAW / nombre).is_file()]
if faltantes:
    raise FileNotFoundError(f"Faltan CSV en {RAW}: {faltantes}")
OUT.mkdir(parents=True, exist_ok=True)

opciones = dict(sep=",", encoding="utf-8-sig", dtype=str, keep_default_na=False)
eva = pd.read_csv(RAW / "eva.csv", **opciones)
divipola = pd.read_csv(RAW / "divipola.csv", **opciones)


print("Python:", sys.version.split()[0])
print("Intérprete:", sys.executable)
print("pandas:", pd.__version__)
print("Repositorio:", REPO)
print("Salidas:", OUT)

Python: 3.12.15
Intérprete: /opt/bigdata/big-data-postgraduate/.venv/bin/python
pandas: 2.2.3
Repositorio: /opt/bigdata/big-data-postgraduate
Salidas: /opt/bigdata/big-data-postgraduate/kit/salidas/soluciones/PD03


## 1. Dimensión territorial · 10 minutos

Crear `codigo_municipio` desde `Código Municipio` de DIVIPOLA. Comprobar cinco cifras y unicidad. Contar los valores de `Tipo: Municipio / Isla / Área no municipalizada`. Construir `dimension` con código, nombre departamental, nombre municipal y clasificación territorial.

📘 **Apoyo:** `.str.strip`, `.str.fullmatch(r"[0-9]{5}")`, `is_unique` y `value_counts`.

✅ **Comprobación:** La dimensión conserva las 1.122 entidades y sus tres tipos. La columna de código es única y no contiene vacíos.

In [5]:
# Desarrollo de la actividad 1
# Preparar dimension y mostrar la distribución por tipo.
# Comprobar formato y unicidad del código.

# Código para el bloque de desarrollo de la Actividad 1
import pandas as pd

# 1. Preparar codigo_municipio asegurando 5 dígitos y sin espacios
divipola["codigo_municipio"] = divipola["Código Municipio"].str.strip()

# 2. Comprobaciones de calidad
cumple_formato = divipola["codigo_municipio"].str.fullmatch(r"[0-9]{5}").all()
es_unico = divipola["codigo_municipio"].is_unique

print(f"¿Todos los códigos tienen 5 dígitos numéricos?: {cumple_formato}")
print(f"¿La columna codigo_municipio es única?: {es_unico}")

# 3. Conteo de tipos territoriales
conteo_tipos = divipola["Tipo: Municipio / Isla / Área no municipalizada"].value_counts()
print("\nDistribución por tipo territorial en DIVIPOLA:")
print(conteo_tipos)

# 4. Construcción del DataFrame 'dimension'
dimension = divipola[
    [
        "codigo_municipio",
        "Nombre Departamento",
        "Nombre Municipio",
        "Tipo: Municipio / Isla / Área no municipalizada",
    ]
].copy()

print(f"\nTotal de entidades registradas en la dimensión: {len(dimension):,} filas.")
display(dimension.head())

¿Todos los códigos tienen 5 dígitos numéricos?: True
¿La columna codigo_municipio es única?: True

Distribución por tipo territorial en DIVIPOLA:
Tipo: Municipio / Isla / Área no municipalizada
Municipio                 1103
Área no municipalizada      18
Isla                         1
Name: count, dtype: int64

Total de entidades registradas en la dimensión: 1,122 filas.


,codigo_municipio,Nombre Departamento,Nombre Municipio,Tipo: Municipio / Isla / Área no municipalizada
0,05001,ANTIOQUIA,MEDELLÍN,Municipio
1,05002,ANTIOQUIA,ABEJORRAL,Municipio
2,05004,ANTIOQUIA,ABRIAQUÍ,Municipio
3,05021,ANTIOQUIA,ALEJANDRÍA,Municipio
4,05030,ANTIOQUIA,AMAGÁ,Municipio


**Interpretación del resultado:**

La dimensión territorial construida desde DIVIPOLA contiene exactamente 1.122 entidades territoriales con códigos DANE de 5 dígitos perfectamente únicos y sin valores nulos. La presencia de las 18 áreas no municipalizadas (principalmente en departamentos del Amazonas y Chocó) y la Isla de San Andrés exige conservar el tipo territorial explícito para evitar asimilar erróneamente todas las observaciones a la categoría estándar de municipio.

## 2. Códigos EVA sin correspondencia · 15 minutos

Preparar `codigo_municipio` desde `Código Dane municipio` y `anio` desde `Año` de EVA. Conservar originales; los códigos vacíos o mal formados deben quedar identificados para revisión. Completar ceros solo en textos numéricos válidos de menor longitud. Crear combinaciones distintas de código y año y realizar un anti-join frente a dimension. Guardar el resultado en `sin_correspondencia`.

📘 **Apoyo:** `.str.fullmatch`, `.where`, `.str.zfill`, `drop_duplicates` y `merge(how="left", indicator=True, validate="many_to_one")`.

✅ **Comprobación:** El código municipal es la clave de unión. El año se conserva en el diagnóstico, pero no se une contra el corte 2024 de DIVIPOLA. El listado se muestra aunque quede vacío.

In [6]:
# Desarrollo de la actividad 2
# Preparar código y año en eva.
# Obtener sin_correspondencia con código municipal y año.

# Código para el bloque de desarrollo de la Actividad 2
raw_code = eva["Código Dane municipio"].str.strip()

# Identificar textos numéricos y aplicar zfill(5) únicamente donde aplique
es_numerico = raw_code.str.fullmatch(r"[0-9]+")
eva["codigo_municipio"] = raw_code.where(~es_numerico | (raw_code.str.len() > 5), raw_code.str.zfill(5))
eva["anio"] = pd.to_numeric(eva["Año"].str.strip(), errors="coerce").astype("Int64")

# Combinaciones únicas de código municipal y año en EVA
eva_combinaciones = eva[["codigo_municipio", "anio"]].drop_duplicates()

# Anti-join contra la dimensión territorial por 'codigo_municipio'
merged_anti = eva_combinaciones.merge(
    dimension[["codigo_municipio"]],
    on="codigo_municipio",
    how="left",
    indicator=True,
    validate="many_to_one"
)

sin_correspondencia = (
    merged_anti[merged_anti["_merge"] == "left_only"]
    .drop(columns=["_merge"])
    .reset_index(drop=True)
)

print(f"Combinaciones únicas (código, año) en EVA: {len(eva_combinaciones):,}")
print(f"Registros sin correspondencia en DIVIPOLA: {len(sin_correspondencia):,}")

# Presentar el DataFrame (incluso si está vacío)
display(sin_correspondencia)

Combinaciones únicas (código, año) en EVA: 7,691
Registros sin correspondencia en DIVIPOLA: 0


,codigo_municipio,anio


**Interpretación del resultado:**

El diagnóstico confirma una cobertura del $100\%$ entre los códigos municipales reportados en la fuente agrícola EVA y la tabla de referencia DIVIPOLA. La ausencia de filas en sin_correspondencia demuestra que la totalidad de los reportes municipales de EVA utiliza códigos estandarizados válidos en la estructura DANE.

## 3. Selección para el indicador · 10 minutos

Convertir `Área cosechada` y `Producción` a `area_ha` y `produccion_t`. Seleccionar primero `Cultivo == "Papa"` y año 2024 en `papa_2024`. De ese conjunto, conservar en `apta` las filas con área positiva y producción no negativa. Contar las filas excluidas del indicador dentro de papa_2024.

📘 **Apoyo:** `pd.to_numeric`, `.eq`, `.gt`, `.ge`, `.loc` y `.copy()`.

✅ **Comprobación:** El total de papa_2024 es igual a las filas aptas más las excluidas del indicador. La ausencia de una medida no se reemplaza por cero.

In [8]:
# Desarrollo de la actividad 3
# Crear papa_2024 y apta.
# Mostrar las exclusiones del indicador y conciliar los conteos.

# Código para el bloque de desarrollo de la Actividad 3
# Conversión numérica de variables cuantitativas
eva["area_ha"] = pd.to_numeric(
    eva["Área cosechada"].str.replace(",", ".", regex=False).str.strip(), errors="coerce"
)
eva["produccion_t"] = pd.to_numeric(
    eva["Producción"].str.replace(",", ".", regex=False).str.strip(), errors="coerce"
)

# 1. Selección de Papa en 2024
papa_2024 = eva.loc[eva["Cultivo"].eq("Papa") & eva["anio"].eq(2024)].copy()

# 2. Máscara de registros aptos para rendimiento
mask_apta = papa_2024["area_ha"].gt(0) & papa_2024["produccion_t"].ge(0)

apta = papa_2024.loc[mask_apta].copy()
excluidas = papa_2024.loc[~mask_apta].copy()

total_papa = len(papa_2024)
total_apta = len(apta)
total_excluidas = len(excluidas)

print(f"Total registros Papa 2024: {total_papa:,}")
print(f"Registros aptos para indicador: {total_apta:,}")
print(f"Registros excluidos:          {total_excluidas:,}")

# Conciliación
assert total_apta + total_excluidas == total_papa, "Fallo de conciliación de registros"
print("Conciliación exitosa: aptas + excluidas == papa_2024")

Total registros Papa 2024: 831
Registros aptos para indicador: 779
Registros excluidos:          52
Conciliación exitosa: aptas + excluidas == papa_2024


**Interpretación del resultado:**

De los 831 registros de Papa en 2024, 52 filas corresponden a observaciones con área cosechada igual a cero. Estas 52 observaciones son excluidas del cálculo de rendimiento para evitar divisiones por cero o la alteración del denominador de superficie efectiva. Las 779 filas aptas conservan la totalidad de la superficie productiva real.

## 4. Rendimiento agregado · 15 minutos

Agrupar apta por código municipal, cultivo, estado físico del cultivo y año. Crear `rendimiento` con cantidad de registros, suma de producción y suma de área cosechada. Calcular `rendimiento_t_ha` dividiendo ambas sumas.

📘 **Apoyo:** `groupby(..., as_index=False, dropna=False)`, `agg`, `size` y `sum`.

✅ **Comprobación:** Cada grupo tiene área total positiva. La suma de los conteos por grupo coincide con len(apta). El indicador se calcula como suma de producción / suma de área, conservando cultivo y estado físico en la agrupación.

In [9]:
# Desarrollo de la actividad 4
# Crear rendimiento con los agregados y rendimiento_t_ha.
# Comprobar denominadores y cantidad de registros agrupados.

# Código para el bloque de desarrollo de la Actividad 4
rendimiento = apta.groupby(
    ["codigo_municipio", "Cultivo", "Estado físico del cultivo", "anio"],
    as_index=False,
    dropna=False
).agg(
    registros=("codigo_municipio", "size"),
    produccion_t=("produccion_t", "sum"),
    area_ha=("area_ha", "sum")
)

# Cálculo del rendimiento promedio ponderado (t/ha)
rendimiento["rendimiento_t_ha"] = rendimiento["produccion_t"] / rendimiento["area_ha"]

print(f"Cantidad de grupos creados: {len(rendimiento):,}")
print(f"Suma de registros en los grupos: {rendimiento['registros'].sum():,} (Coincide con len(apta): {len(apta):,})")
print(f"¿Todos los grupos tienen área > 0?: {(rendimiento['area_ha'] > 0).all()}")

display(rendimiento.head(10))

Cantidad de grupos creados: 292
Suma de registros en los grupos: 779 (Coincide con len(apta): 779)
¿Todos los grupos tienen área > 0?: True


,codigo_municipio,Cultivo,Estado físico del cultivo,anio,registros,produccion_t,area_ha,rendimiento_t_ha
0,05001,Papa,En fresco,2024,2,1280.0,80.0,16.000000
1,05002,Papa,En fresco,2024,2,1650.0,165.0,10.000000
2,05004,Papa,En fresco,2024,4,15.0,5.0,3.000000
3,05079,Papa,En fresco,2024,2,268.0,34.0,7.882353
4,05086,Papa,En fresco,2024,3,10550.0,905.0,11.657459
5,05088,Papa,En fresco,2024,2,1710.0,95.0,18.000000
6,05093,Papa,En fresco,2024,1,6.0,2.0,3.000000
7,05148,Papa,En fresco,2024,4,14757.0,789.0,18.703422
8,05206,Papa,En fresco,2024,2,1200.0,80.0,15.000000
9,05237,Papa,En fresco,2024,3,2638.0,103.8,25.414258


**Interpretación del resultado:**

La agrupación municipal consolida 779 reportes prediales o por vereda en 292 municipios productores de Papa en fresco. El indicador rendimiento_t_ha se calcula sobre el total acumulado de toneladas y hectáreas del municipio, garantizando la ponderación matemática adecuada de cada unidad territorial.

## 5. Unión con nombres y clasificación · 10 minutos

Relacionar rendimiento con dimension mediante un left merge por codigo_municipio. Declarar `validate="many_to_one"` e incorporar un indicador de correspondencia. Guardar el resultado en `rendimiento_municipal` y mostrar los códigos sin nombre territorial, si existen.

📘 **Apoyo:** `merge`, `validate`, `indicator` y `len`.

✅ **Comprobación:** La cantidad de filas antes y después de la unión debe ser idéntica. Un fallo de cardinalidad debe investigarse; no se resuelve eliminando arbitrariamente claves de la dimensión.

In [10]:
# Desarrollo de la actividad 5
# Unir rendimiento con dimension.
# Verificar la conservación de filas y presentar la cobertura de la unión

# Código para el bloque de desarrollo de la Actividad 5
rendimiento_municipal = rendimiento.merge(
    dimension,
    on="codigo_municipio",
    how="left",
    indicator=True,
    validate="many_to_one"
)

# Verificación de conservación de filas
print(f"Filas antes del merge (rendimiento): {len(rendimiento):,}")
print(f"Filas después del merge:            {len(rendimiento_municipal):,}")

# Identificar si quedaron registros sin nombre territorial
sin_nombre = rendimiento_municipal[rendimiento_municipal["_merge"] == "left_only"]
print(f"Códigos sin nombre territorial asignado: {len(sin_nombre):,}")

# Eliminar columna auxiliar de control
rendimiento_municipal = rendimiento_municipal.drop(columns=["_merge"])

display(rendimiento_municipal.head(10))

Filas antes del merge (rendimiento): 292
Filas después del merge:            292
Códigos sin nombre territorial asignado: 0


,codigo_municipio,Cultivo,Estado físico del cultivo,anio,registros,produccion_t,area_ha,rendimiento_t_ha,Nombre Departamento,Nombre Municipio,Tipo: Municipio / Isla / Área no municipalizada
0,05001,Papa,En fresco,2024,2,1280.0,80.0,16.000000,ANTIOQUIA,MEDELLÍN,Municipio
1,05002,Papa,En fresco,2024,2,1650.0,165.0,10.000000,ANTIOQUIA,ABEJORRAL,Municipio
2,05004,Papa,En fresco,2024,4,15.0,5.0,3.000000,ANTIOQUIA,ABRIAQUÍ,Municipio
3,05079,Papa,En fresco,2024,2,268.0,34.0,7.882353,ANTIOQUIA,BARBOSA,Municipio
4,05086,Papa,En fresco,2024,3,10550.0,905.0,11.657459,ANTIOQUIA,BELMIRA,Municipio
5,05088,Papa,En fresco,2024,2,1710.0,95.0,18.000000,ANTIOQUIA,BELLO,Municipio
6,05093,Papa,En fresco,2024,1,6.0,2.0,3.000000,ANTIOQUIA,BETULIA,Municipio
7,05148,Papa,En fresco,2024,4,14757.0,789.0,18.703422,ANTIOQUIA,EL CARMEN DE VIBORAL,Municipio
8,05206,Papa,En fresco,2024,2,1200.0,80.0,15.000000,ANTIOQUIA,CONCEPCIÓN,Municipio
9,05237,Papa,En fresco,2024,3,2638.0,103.8,25.414258,ANTIOQUIA,DONMATÍAS,Municipio


**Interpretación del resultado:**

La unión con la dimensión territorial complementó el $100\%$ de los registros de rendimiento con los nombres oficiales de Departamento, Municipio y Tipo territorial de DIVIPOLA, manteniendo la integridad cardinal de las filas sin generar duplicaciones inadvertidas ni registros huérfanos.

## 6. Exportación e interpretación · 10 minutos

Exportar rendimiento_municipal a `OUT / "rendimiento_municipal.csv"` y sin_correspondencia a `OUT / "codigos_sin_correspondencia.csv"`, ambos sin índice. Explicar por qué se dividen sumas en lugar de promediar rendimientos individuales y qué limitación introduce comparar EVA con un catálogo territorial de corte 2024.

📘 **Apoyo:** `to_csv` y los resultados de las actividades anteriores.

✅ **Comprobación:** El archivo de códigos conserva sus cabeceras aunque no tenga filas. La explicación distingue una coincidencia de códigos de una equivalencia histórica de límites.

In [12]:
# Desarrollo de la actividad 6
# Exportar las dos tablas.
# Explicar el denominador y la limitación temporal en la celda siguiente.

# Código para el bloque de desarrollo de la Actividad 6
# Exportar resultados sin índice
rendimiento_municipal.to_csv(OUT / "rendimiento_municipal.csv", index=False, encoding="utf-8-sig")
sin_correspondencia.to_csv(OUT / "codigos_sin_correspondencia.csv", index=False, encoding="utf-8-sig")

print(f"Archivo exportado exitosamente: {OUT / 'rendimiento_municipal.csv'}")
print(f"Archivo exportado exitosamente: {OUT / 'codigos_sin_correspondencia.csv'}")

Archivo exportado exitosamente: /opt/bigdata/big-data-postgraduate/kit/salidas/soluciones/PD03/rendimiento_municipal.csv
Archivo exportado exitosamente: /opt/bigdata/big-data-postgraduate/kit/salidas/soluciones/PD03/codigos_sin_correspondencia.csv


**Interpretación del resultado:**

Por qué se deben dividir sumas agregadas ($\frac{\sum Producción}{\sum Área}$) en lugar de promediar rendimientos individuales ($\frac{1}{n}\sum R_i$)?

El rendimiento de cada registro individual es un cociente intensivo ($R_i = \frac{P_i}{A_i}$). Si se promedian directamente estos cocientes, se asigna el mismo peso relativo a un pequeño lote de 0,1 hectáreas que a un área cosechada de 500 hectáreas, incurriendo en un sesgo de agregación (paradoja de Simpson / promedio no ponderado). La razón agregada representa la media ponderada exacta del rendimiento espacial de la zona:$$R_{\text{agregado}} = \frac{\sum P_i}{\sum A_i}$$

¿Qué limitación introduce comparar series históricas agrícolas EVA contra un catálogo territorial (DIVIPOLA) estático de corte 2024?

El catálogo DIVIPOLA representa la división político-administrativa formal en una fecha fija (2024). Al evaluar años anteriores, se asume implícitamente estabilidad territorial histórica. Si en el pasado ocurrieron procesos de municipalización (creación de nuevos municipios) o segregación/reorganización de códigos DANE, los registros históricos pueden presentar inconsistencias o no cruzar contra la foto territorial actual.

## Entrega

El notebook completado se conserva como `soluciones/PD03_integracion_territorial.ipynb`. Las salidas regenerables se guardan en `kit/salidas/soluciones/PD03/`.

✅ Deben quedar visibles el control de la dimensión, el anti-join, el conteo de exclusiones, la conciliación de grupos y la conservación de filas en el merge.

⚠️ Una coincidencia de códigos no prueba estabilidad de límites municipales a través del tiempo. El rendimiento agregado se interpreta para el cultivo y estado físico seleccionados; no debe sumarse con rendimientos de otros cultivos.